# Database Build

This notebook loads cleaned transit and BART datasets into SQLite, creates analysis tables and views, and validates the database structure for downstream analysis.

## Section 1: Imports and Database Connection

In [92]:
import pandas as pd
import sqlite3
import os 
os.makedirs('../database', exist_ok=True)
conn = sqlite3.connect('../database/bay_area_transit.db')

## Section 2: Load Cleaned Data

In [93]:
commute_df = pd.read_csv('../data/processed/commute_times_clean.csv')

bart_df = pd.read_csv('../data/processed/bart_stations_clean.csv')
print(commute_df.shape)
print(bart_df.shape)

(60, 10)
(50, 6)


## Section 3: Load Tables into SQLite

In [94]:
commute_df.to_sql('commute_times',conn,if_exists='replace',index=False)

bart_df.to_sql('bart_stations', conn, if_exists='replace', index=False)

query= """
SELECT name
FROM sqlite_master
WHERE type='table'
"""

pd.read_sql(query,conn)

,name
0,station_city_lookup
1,city_ridership
2,commute_times
3,bart_stations


## Section 4: Create City Lookup Table

In [95]:
lookup = pd.read_csv('../data/raw/station_city_lookup.csv')

lookup.to_sql('station_city_lookup',conn,if_exists='replace',index=False)

50

## Section 5: Create city level ridership

In [96]:
query = """
DROP TABLE IF EXISTS city_ridership;
CREATE TABLE city_ridership AS
SELECT
    l.city,
    SUM(b.avg_weekday_riders) AS avg_weekday_riders,
    COUNT(b.station) AS num_stations
FROM bart_stations b
JOIN station_city_lookup l
ON b.station = l.station
GROUP BY l.city;
"""
conn.executescript(query)
pd.read_sql("SELECT * FROM city_ridership LIMIT 10", conn)


,city,avg_weekday_riders,num_stations
0,Antioch,1876,1
1,Berkeley,9841,3
2,Castro Valley,1313,1
3,Colma,1741,1
4,Concord,3344,2
5,Daly City,4750,1
6,Dublin,1343,1
7,El Cerrito,6358,2
8,Fremont,3312,2
9,Hayward,3670,2


## Section 6: Create Analysis View

In [97]:
query = """
DROP VIEW IF EXISTS transit_analysis;

CREATE VIEW transit_analysis AS
SELECT
    c.jurisdiction,
    c.county,
    c.drive_alone,
    c.carpool,
    c.transit,

    c.transit - c.drive_alone AS transit_penalty,
    c.carpool - c.drive_alone AS carpool_penalty,

    r.avg_weekday_riders,
    r.num_stations,

    CAST(r.avg_weekday_riders AS FLOAT) / r.num_stations AS riders_per_station

FROM commute_times c
LEFT JOIN city_ridership r
ON c.jurisdiction = r.city;
"""

conn.executescript(query)

## Section 7: Validation

In [98]:
query = """
SELECT *
FROM transit_analysis
LIMIT 10
"""

pd.read_sql(query,conn)

query = """
SELECT COUNT(*)
FROM transit_analysis
"""

pd.read_sql(query, conn)

# 1. Confirm all expected tables and views exist
query = """
SELECT name, type
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name;
"""

pd.read_sql(query, conn)

# 2. Confirm row counts for each table/view
queries = {
    "commute_times": "SELECT COUNT(*) AS row_count FROM commute_times",
    "bart_stations": "SELECT COUNT(*) AS row_count FROM bart_stations",
    "station_city_lookup": "SELECT COUNT(*) AS row_count FROM station_city_lookup",
    "city_ridership": "SELECT COUNT(*) AS row_count FROM city_ridership",
    "transit_analysis": "SELECT COUNT(*) AS row_count FROM transit_analysis"
}

for name, query in queries.items():
    print(name)
    display(pd.read_sql(query, conn))

# 3. Check for unmatched BART stations in the lookup table
query = """
SELECT
    b.station
FROM bart_stations b
LEFT JOIN station_city_lookup l
ON b.station = l.station
WHERE l.city IS NULL;
"""

pd.read_sql(query, conn)

# 4. Check for cities with no BART ridership after the left join
query = """
SELECT
    jurisdiction,
    avg_weekday_riders,
    num_stations
FROM transit_analysis
WHERE avg_weekday_riders IS NULL;
"""

pd.read_sql(query, conn)

# 5. Check that calculated fields are working
query = """
SELECT
    jurisdiction,
    drive_alone,
    transit,
    transit_penalty,
    carpool_penalty,
    riders_per_station
FROM transit_analysis
LIMIT 10;
"""

pd.read_sql(query, conn)

# 6. Check summary ranges for key metrics
query = """
SELECT
    COUNT(*) AS rows,
    MIN(transit_penalty) AS min_transit_penalty,
    MAX(transit_penalty) AS max_transit_penalty,
    AVG(transit_penalty) AS avg_transit_penalty,
    MIN(avg_weekday_riders) AS min_riders,
    MAX(avg_weekday_riders) AS max_riders
FROM transit_analysis;
"""

pd.read_sql(query, conn)

conn.close()


commute_times


,row_count
0,60


bart_stations


,row_count
0,50


station_city_lookup


,row_count
0,50


city_ridership


,row_count
0,26


transit_analysis


,row_count
0,60
